In [0]:
def validate_scd2(table_name, natural_key, surrogate_key):
    print(f"=== Validating {table_name} ===")
    all_passed = True

    # Check 1: exactly one is_current = true per natural key
    multi_current = spark.sql(f"""
        SELECT {natural_key}, COUNT(*) AS current_count
        FROM {table_name}
        WHERE is_current = true
        GROUP BY {natural_key}
        HAVING COUNT(*) <> 1
    """)
    cnt = multi_current.count()
    if cnt > 0:
        print(f"FAIL: {cnt} natural key(s) with != 1 current row")
        multi_current.show(20, truncate=False)
        all_passed = False
    else:
        print("PASS: exactly one is_current=true per natural key")

    # Check 2: no overlapping effective date ranges per natural key
    overlaps = spark.sql(f"""
        SELECT a.{natural_key},
               a.{surrogate_key} AS key_a, a.effective_date AS start_a, a.end_date AS end_a,
               b.{surrogate_key} AS key_b, b.effective_date AS start_b, b.end_date AS end_b
        FROM {table_name} a
        JOIN {table_name} b
          ON a.{natural_key} = b.{natural_key}
         AND a.{surrogate_key} <> b.{surrogate_key}
         AND a.effective_date < COALESCE(b.end_date, TIMESTAMP('9999-12-31'))
         AND COALESCE(a.end_date, TIMESTAMP('9999-12-31')) > b.effective_date
    """)
    cnt = overlaps.count()
    if cnt > 0:
        print(f"FAIL: {cnt} overlapping effective date range pair(s)")
        overlaps.show(20, truncate=False)
        all_passed = False
    else:
        print("PASS: no overlapping effective date ranges")

    # Check 3: surrogate keys unique
    dup_keys = spark.sql(f"""
        SELECT {surrogate_key}, COUNT(*) AS dup_count
        FROM {table_name}
        GROUP BY {surrogate_key}
        HAVING COUNT(*) > 1
    """)
    cnt = dup_keys.count()
    if cnt > 0:
        print(f"FAIL: {cnt} duplicate surrogate key(s)")
        dup_keys.show(20, truncate=False)
        all_passed = False
    else:
        print("PASS: surrogate keys unique")

    print(f"=== {table_name}: {'ALL CHECKS PASSED' if all_passed else 'FAILURES FOUND'} ===\n")
    return all_passed

In [0]:
seller_ok  = validate_scd2("ecommerce_dev.gold.dim_seller", "seller_id", "seller_key")
product_ok = validate_scd2("ecommerce_dev.gold.dim_product", "product_id", "product_key")

assert seller_ok and product_ok, "SCD2 validation failed — see output above"

=== Validating ecommerce_dev.gold.dim_seller ===
PASS: exactly one is_current=true per natural key
PASS: no overlapping effective date ranges
PASS: surrogate keys unique
=== ecommerce_dev.gold.dim_seller: ALL CHECKS PASSED ===

=== Validating ecommerce_dev.gold.dim_product ===
PASS: exactly one is_current=true per natural key
PASS: no overlapping effective date ranges
PASS: surrogate keys unique
=== ecommerce_dev.gold.dim_product: ALL CHECKS PASSED ===

